# Mini Project: Image Classification with Data Augmentation (Cats vs Dogs)
## Week 6 — Day 4 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build a binary CNN classifier to distinguish cats from dogs. Practice end-to-end deep learning: data pipelines, augmentation, model design, evaluation, inference, and ablation.

**Sections:**
1. Data Loading & Generators
2. Inspect the Data
3. Define the Model Architecture
4. Choose the Optimization Setup
5. Train the Model
6. Evaluate on Validation Data
7. Run Inference on the Unlabeled Test Set
8. Compare Baseline vs Augmentation
9. Class Imbalance Handling
10. Save Artifacts for Reuse
11. Extensions (Transfer Learning with MobileNetV2)
12. Deliverables Checklist

> **Run on Google Colab / DigitalOcean VM with GPU enabled.**

## Setup — Download Dataset

Download the Kaggle Dogs vs Cats dataset, extract it, and rename the folder to match the expected layout.  
**You need a Kaggle API key.** Upload `kaggle.json` to Colab or set credentials as below.

In [ ]:
import os

# --- Option A: Upload kaggle.json via Colab file upload ---
# from google.colab import files
# files.upload()  # upload your kaggle.json

# --- Option B: Paste credentials directly ---
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'

# Setup kaggle directory
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
# If you used Option A:
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!pip install kaggle --quiet
!kaggle competitions download -c dogs-vs-cats -p /content/ --quiet

# Extract
import zipfile
with zipfile.ZipFile('/content/dogs-vs-cats.zip', 'r') as z:
    z.extractall('/content/')
with zipfile.ZipFile('/content/train.zip', 'r') as z:
    z.extractall('/content/')
with zipfile.ZipFile('/content/test1.zip', 'r') as z:
    z.extractall('/content/')

# Create expected folder structure: data/cats_dogs/train/train/ and data/cats_dogs/test/test/
import shutil
os.makedirs('data/cats_dogs/train/train', exist_ok=True)
os.makedirs('data/cats_dogs/test/test',  exist_ok=True)

shutil.copytree('/content/train', 'data/cats_dogs/train/train', dirs_exist_ok=True)
shutil.copytree('/content/test1', 'data/cats_dogs/test/test',   dirs_exist_ok=True)

print("Dataset ready ✓")
print(f"  Train images: {len(os.listdir('data/cats_dogs/train/train'))}")
print(f"  Test  images: {len(os.listdir('data/cats_dogs/test/test'))}")

## Part 1 — Data Loading & Generators (Prefilled)

In [ ]:
# Prefilled. Just copy and execute.
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42); tf.random.set_seed(42)

# Paths - change if needed
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

# Build DataFrames from folders
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"No images found under {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat","cats"}:
                label = "cat"
            elif parent in {"dog","dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else:
                    continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Train validation split
from sklearn.model_selection import train_test_split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)

# Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=False, validate_filenames=False
)
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

print({"train": train_flow.samples, "val": val_flow.samples, "test": test_flow.samples,
       "class_indices": train_flow.class_indices})

## Part 2 — Inspect the Data

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# --- Class balance report ---
label_map    = {v: k for k, v in train_flow.class_indices.items()}  # {0: 'cat', 1: 'dog'}
labels_arr   = train_flow.labels  # 0/1 array
n_cat_train  = int(np.sum(labels_arr == train_flow.class_indices['cat']))
n_dog_train  = int(np.sum(labels_arr == train_flow.class_indices['dog']))
n_total      = len(labels_arr)

print("=" * 40)
print("  CLASS BALANCE REPORT")
print("=" * 40)
print(f"  class_indices : {train_flow.class_indices}")
print(f"  cat  (train)  : {n_cat_train:,}  ({n_cat_train/n_total*100:.1f}%)")
print(f"  dog  (train)  : {n_dog_train:,}  ({n_dog_train/n_total*100:.1f}%)")
print(f"  Total         : {n_total:,}")
print("=" * 40)
print("\n  The Kaggle Dogs vs Cats dataset contains 12,500 images per class —")
print("  a perfectly balanced 50/50 split. No class weighting is needed.")

# Balance bar chart
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Cat', 'Dog'], [n_cat_train, n_dog_train], color=['#4e91d4', '#e05c5c'])
ax.set_title('Class Distribution — Training Set', fontweight='bold')
ax.set_ylabel('Number of Images')
for i, v in enumerate([n_cat_train, n_dog_train]):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('cvd_plot1_class_balance.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 1 saved ✓")

In [ ]:
# --- Sample image grid (4×4, labeled) ---
x_batch, y_batch = next(iter(
    val_gen.flow_from_dataframe(
        df_val, x_col="filepath", y_col="label",
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        class_mode="binary", batch_size=16,
        shuffle=True, seed=42, validate_filenames=False
    )
))

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_batch[i])
    cls = label_map[int(y_batch[i])]
    color = '#4e91d4' if cls == 'cat' else '#e05c5c'
    ax.set_title(cls.upper(), color=color, fontweight='bold', fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Training Images (rescaled only, no augmentation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cvd_plot2_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 2 saved ✓")

print("""
Visual analysis:
  CATS: pointed ears, vertical whiskers, narrow muzzle, slit pupils, smaller face.
  DOGS: floppy/upright ears, broader snout, round pupils, larger head relative to body.
  Sources of variability:
    • Pose (side view, front, sleeping, mid-action)
    • Scale (close-up portrait vs full body)
    • Lighting (indoor flash, outdoor sunlight, shadow)
    • Background clutter (grass, furniture, other animals)
  These factors drive the need for rotation, zoom, and shift augmentation.
""")

## Part 3 — Define the Model Architecture

**Architecture description:**

The CNN has **3 convolutional blocks**, each consisting of a `Conv2D` layer followed by `BatchNormalization`, `ReLU` activation, and `MaxPooling2D`. Filter counts double with depth (32 → 64 → 128) to capture increasingly abstract features — edges and textures in early layers, shapes and parts in deeper ones.

After the convolutional blocks, `GlobalAveragePooling2D` replaces a large `Flatten` layer, reducing parameters and spatial overfitting. The classifier head has one `Dense(256, ReLU)` layer with `Dropout(0.5)` before the final `Dense(1, sigmoid)`.

`Sigmoid` is correct for binary targets: it outputs a probability ∈ (0,1) and paired with `binary_crossentropy` it maximizes the log-likelihood of the correct Bernoulli label. `Softmax` is reserved for mutually-exclusive multi-class problems.

In [ ]:
from tensorflow.keras import layers, models

def build_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), dropout=0.5):
    model = models.Sequential([
        # Block 1 — 32 filters
        layers.Conv2D(32, (3,3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),

        # Block 2 — 64 filters
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),

        # Block 3 — 128 filters
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),

        # Block 4 — 256 filters
        layers.Conv2D(256, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.GlobalAveragePooling2D(),

        # Classifier head
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid')
    ], name='cats_dogs_cnn')
    return model

model = build_cnn()
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

## Part 4 — Choose the Optimization Setup

**Choices and justifications:**
- **Optimizer: Adam (lr=1e-4)** — Adaptive per-parameter learning rates converge faster than SGD. `1e-4` is conservative enough to avoid overshooting on the relatively small dataset.
- **Loss: `binary_crossentropy`** — Correct for Bernoulli targets with sigmoid output; measures log-probability of the correct label.
- **Metric: `accuracy`** — Dataset is balanced so accuracy is meaningful; loss is still the primary signal.
- **Batch size: 32** — Balances GPU memory with gradient noise; larger batches can miss sharp minima.
- **EarlyStopping (patience=5)** — Stops when `val_loss` stops improving for 5 consecutive epochs, restoring best weights.
- **ReduceLROnPlateau (patience=3, factor=0.5)** — Halves learning rate when validation stalls, helping escape plateaus.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

LEARNING_RATE = 1e-4
EPOCHS        = 30

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_cats_dogs.h5', monitor='val_accuracy', save_best_only=True, verbose=0)
]

print("Optimization setup ready ✓")
print(f"  Optimizer : Adam (lr={LEARNING_RATE})")
print(f"  Loss      : binary_crossentropy")
print(f"  Epochs    : up to {EPOCHS} (EarlyStopping patience=5)")

## Part 5 — Train the Model

In [ ]:
steps_per_epoch  = math.ceil(train_flow.samples / batch_size)
validation_steps = math.ceil(val_flow.samples   / batch_size)

print(f"Training augmented model...")
print(f"  Steps/epoch : {steps_per_epoch}  |  Val steps: {validation_steps}")

history = model.fit(
    train_flow,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_flow,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)
print("\nTraining complete ✓")

In [ ]:
# Training & validation curves
def plot_history(hist, title='Training History', save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs_ran = range(1, len(hist.history['accuracy']) + 1)

    axes[0].plot(epochs_ran, hist.history['accuracy'],     label='Train', color='#4e91d4', lw=2)
    axes[0].plot(epochs_ran, hist.history['val_accuracy'], label='Val',   color='#e05c5c', lw=2, linestyle='--')
    axes[0].set_title('Accuracy', fontweight='bold'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_ran, hist.history['loss'],     label='Train', color='#4e91d4', lw=2)
    axes[1].plot(epochs_ran, hist.history['val_loss'], label='Val',   color='#e05c5c', lw=2, linestyle='--')
    axes[1].set_title('Loss', fontweight='bold'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Binary Crossentropy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

plot_history(history, title='Augmented CNN — Training History', save_path='cvd_plot3_aug_curves.png')
print("Plot 3 saved ✓")

# Detect overfitting
train_acc_final = history.history['accuracy'][-1]
val_acc_final   = history.history['val_accuracy'][-1]
gap = train_acc_final - val_acc_final
print(f"\nFinal train acc : {train_acc_final*100:.2f}%")
print(f"Final val   acc : {val_acc_final*100:.2f}%")
print(f"Generalization gap : {gap*100:.2f}%")
if gap > 0.10:
    print("  → Overfitting detected. Mitigations: stronger dropout, more augmentation, fewer filters.")
else:
    print("  → Model generalizes well. Gap is small.")

## Part 6 — Evaluate on Validation Data

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Re-run val_flow from scratch to get all labels aligned
val_flow.reset()
y_probs = model.predict(val_flow, steps=validation_steps, verbose=1)
y_pred  = (y_probs.flatten() > 0.5).astype(int)
y_true  = val_flow.labels[:len(y_pred)]

val_loss, val_acc = model.evaluate(val_flow, steps=validation_steps, verbose=0)
print(f"\nValidation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc*100:.2f}%")

# Confusion matrix
CLASSES = ['cat', 'dog']
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=1, ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# Per-class accuracy bar
report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
prec = [report['cat']['precision'], report['dog']['precision']]
rec  = [report['cat']['recall'],    report['dog']['recall']]
x = np.arange(2)
axes[1].bar(x - 0.2, prec, 0.35, label='Precision', color='#4e91d4')
axes[1].bar(x + 0.2, rec,  0.35, label='Recall',    color='#e05c5c')
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASSES)
axes[1].set_ylim(0, 1.1); axes[1].set_ylabel('Score')
axes[1].set_title('Precision & Recall per Class', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Validation Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cvd_plot4_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 4 saved ✓")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES))
print("Analysis: If recall for 'cat' < recall for 'dog', the model has a bias toward predicting")
print("dog — possibly because dogs have more varied fur textures visible to conv filters.")
print("Adjusting the decision threshold below 0.5 would increase cat recall at cost of precision.")

## Part 7 — Run Inference on the Unlabeled Test Set

In [ ]:
THRESHOLD = 0.5  # 0.5 is standard; lower to improve cat recall if needed

test_flow.reset()
test_steps  = math.ceil(test_flow.samples / batch_size)
test_probs  = model.predict(test_flow, steps=test_steps, verbose=1).flatten()
test_probs  = test_probs[:test_flow.samples]  # trim padding from last batch
test_labels = ['dog' if p > THRESHOLD else 'cat' for p in test_probs]
test_paths  = test_flow.filenames

results_df = pd.DataFrame({
    'filepath'  : test_paths,
    'prob_dog'  : test_probs.round(4),
    'pred_label': test_labels
})

results_df.to_csv('test_predictions.csv', index=False)
print(f"Predictions saved → test_predictions.csv  ({len(results_df)} rows)")
print(f"\nPredicted distribution:")
print(results_df['pred_label'].value_counts())
print(f"\nHigh-confidence predictions (prob > 0.9 or < 0.1): {((test_probs > 0.9) | (test_probs < 0.1)).sum()}")
print(f"Uncertain predictions (0.4 < prob < 0.6)          : {((test_probs > 0.4) & (test_probs < 0.6)).sum()}")

# Preview
print("\nFirst 10 predictions:")
display(results_df.head(10))

print("\nManual verification strategy:")
print("  1. Sample ~50 random rows from the CSV and visually inspect the actual images.")
print("  2. Focus especially on predictions with prob_dog ∈ [0.4, 0.6] — the uncertain zone.")
print("  3. If a systematic error is found (e.g. white cats → dog), add targeted augmentation.")
print(f"  4. Threshold={THRESHOLD} is arbitrary. Lower it to bias toward predicting 'dog' less.")

## Part 8 — Compare Baseline vs Augmentation

In [ ]:
# Baseline: same architecture, NO augmentation
baseline_gen = ImageDataGenerator(rescale=1./255)
baseline_flow = baseline_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)

model_baseline = build_cnn()
model_baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

baseline_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)
]

print("Training BASELINE model (no augmentation)...")
history_baseline = model_baseline.fit(
    baseline_flow,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_flow,
    validation_steps=validation_steps,
    callbacks=baseline_callbacks,
    verbose=1
)
print("Baseline training complete ✓")

In [ ]:
# Compare curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'baseline': '#f0ad4e', 'augmented': '#4e91d4'}

for hist, name in [(history_baseline, 'baseline'), (history, 'augmented')]:
    ep = range(1, len(hist.history['accuracy']) + 1)
    axes[0].plot(ep, hist.history['val_accuracy'], label=name, color=colors[name], lw=2)
    axes[1].plot(ep, hist.history['val_loss'],     label=name, color=colors[name], lw=2)

for ax, title in zip(axes, ['Val Accuracy', 'Val Loss']):
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Baseline vs Augmented — Validation Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cvd_plot5_ablation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 5 saved ✓")

# Final metrics comparison
_, bl_val_acc  = model_baseline.evaluate(val_flow, steps=validation_steps, verbose=0)
_, aug_val_acc = model.evaluate(val_flow,           steps=validation_steps, verbose=0)

bl_gap  = history_baseline.history['accuracy'][-1] - bl_val_acc
aug_gap = history.history['accuracy'][-1]           - aug_val_acc

print(f"\n{'='*50}")
print(f"  {'MODEL':<20} {'Val Acc':>10} {'Gap':>10}")
print(f"{'='*50}")
print(f"  {'Baseline (no aug)':<20} {bl_val_acc*100:>9.2f}% {bl_gap*100:>9.2f}%")
print(f"  {'Augmented':<20} {aug_val_acc*100:>9.2f}% {aug_gap*100:>9.2f}%")
print(f"{'='*50}")
print("\nAnalysis:")
print("  Baseline trains faster but shows a larger generalization gap (overfits).")
print("  Augmentation increases effective dataset size → smaller gap, better robustness.")
print("  The improvement is cost-free: same architecture, same parameters, same compute.")

## Part 9 — Class Imbalance Handling

The Kaggle Dogs vs Cats dataset is perfectly balanced (12,500 images per class), so class weighting is not strictly needed. However, if we were working with an imbalanced subset, here's how to apply it.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights from training labels
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_flow.labels
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print("Class weights (for reference):")
for cls_idx, cls_name in label_map.items():
    print(f"  Class {cls_idx} ({cls_name}): weight = {class_weight_dict[cls_idx]:.4f}")

print(f"\nSince the dataset is balanced (50/50), both weights ≈ 1.0.")
print("If the dataset were 80% dogs / 20% cats, the cat weight would be ~4.0,")
print("making each cat misclassification count 4× more in the loss — compensating for fewer examples.")
print("\nEffect on metrics:")
print("  • Minority class recall ↑ (model tries harder not to miss it)")
print("  • Minority class precision may ↓ slightly (more false positives in minority)")
print("  • Overall accuracy may ↓ slightly but F1 on minority class improves")

# The augmented model already handles this; skipping a full retrain since dataset is balanced
print("\n[No retraining needed — dataset is perfectly balanced]")

## Part 10 — Save Artifacts for Reuse

In [ ]:
import json

# Save in both formats
model.save('cats_dogs_model.h5')
model.save('cats_dogs_savedmodel')

print("Model saved:")
print(f"  cats_dogs_model.h5    — Keras HDF5 (single file, portable)")
print(f"  cats_dogs_savedmodel/ — TF SavedModel (recommended for serving)")

# Save training config as JSON
config = {
    "model_name"      : "cats_dogs_cnn",
    "input_shape"     : [IMG_HEIGHT, IMG_WIDTH, 3],
    "architecture"    : "4x(Conv2D-BN-ReLU-MaxPool) + GAP + Dense(256) + Dropout(0.5) + Dense(1)",
    "optimizer"       : "Adam",
    "learning_rate"   : LEARNING_RATE,
    "loss"            : "binary_crossentropy",
    "batch_size"      : batch_size,
    "augmentation"    : {
        "rotation_range"     : 45,
        "width_shift_range"  : 0.15,
        "height_shift_range" : 0.15,
        "zoom_range"         : 0.5,
        "horizontal_flip"    : True
    },
    "epochs_trained"  : len(history.history['accuracy']),
    "best_val_accuracy": float(max(history.history['val_accuracy'])),
    "class_indices"   : train_flow.class_indices,
    "threshold"       : THRESHOLD
}

with open('training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("\nTraining config saved → training_config.json")
print(json.dumps(config, indent=2))

print("""
Why save both weights AND metadata?
  • Weights alone cannot be reused without knowing the architecture, normalization,
    threshold, and class mapping used during training.
  • If someone loads best_cats_dogs.h5 but applies a different image scale or
    threshold, predictions will be wrong silently.
  • The JSON records every choice so any team member can reproduce results exactly.
""")

## Part 11 — Extensions: Transfer Learning with MobileNetV2

**Justification:** MobileNetV2 was pretrained on ImageNet (1.4M images, 1000 classes). Its early layers already detect edges, textures, and shapes common to all natural images. By freezing these layers and training only a small classification head, we get strong feature representations with far fewer training examples and epochs.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Input, Model

# Load MobileNetV2 without top classifier, pretrained on ImageNet
base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze backbone — use pretrained features as-is
print(f"MobileNetV2 loaded: {base_model.count_params():,} params (all frozen)")

# Add a lightweight classification head
inputs  = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model_tl = Model(inputs, outputs, name='mobilenetv2_cats_dogs')

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

trainable_params = sum(p.numpy().size for p in model_tl.trainable_weights)
print(f"Trainable parameters: {trainable_params:,}  (head only)")
model_tl.summary()

# Train head only (fast — backbone frozen)
print("\nTraining MobileNetV2 transfer learning model (head only)...")
tl_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)
]
history_tl = model_tl.fit(
    train_flow, steps_per_epoch=steps_per_epoch, epochs=15,
    validation_data=val_flow, validation_steps=validation_steps,
    callbacks=tl_callbacks, verbose=1
)

_, tl_val_acc = model_tl.evaluate(val_flow, steps=validation_steps, verbose=0)
print(f"\n{'='*45}")
print(f"  Transfer Learning (MobileNetV2) Val Acc: {tl_val_acc*100:.2f}%")
print(f"  Custom CNN Val Acc               : {aug_val_acc*100:.2f}%")
print(f"{'='*45}")
print("Transfer learning typically achieves higher accuracy in fewer epochs because")
print("the frozen backbone provides rich, general-purpose visual features for free.")

## Part 12 — Deliverables Checklist

| # | Deliverable | Status |
|---|-------------|--------|
| 1 | Data report — class counts + sample image grid | ✅ Parts 1 & 2 |
| 2 | Model description & optimization rationale in prose | ✅ Parts 3 & 4 |
| 3 | Training & validation curves with interpretation | ✅ Part 5 |
| 4 | Validation metrics — confusion matrix + precision/recall | ✅ Part 6 |
| 5 | Test predictions CSV (`test_predictions.csv`) | ✅ Part 7 |
| 6 | Saved model files (`cats_dogs_model.h5`, `cats_dogs_savedmodel/`) | ✅ Part 10 |
| 7 | Training config JSON (`training_config.json`) | ✅ Part 10 |
| 8 | Augmentation ablation (baseline vs augmented) | ✅ Part 8 |
| 9 | Class imbalance analysis | ✅ Part 9 |
| 10 | Extension — Transfer learning (MobileNetV2) | ✅ Part 11 |

**Key learnings:**
1. A clean input pipeline (proper label inference + fixed val split) is the highest-leverage step.
2. Data augmentation reduces overfitting for free — same parameters, better generalization.
3. `GlobalAveragePooling2D` replaces `Flatten` to cut parameters and spatial overfitting.
4. The 0.5 sigmoid threshold is arbitrary — calibrate it for the target recall/precision trade-off.
5. Transfer learning (frozen MobileNetV2 backbone) typically outperforms a custom CNN trained from scratch on the same data, using only a fraction of the training time.